# Hotel Bar Inventory Forecasting and Par Level Recommendation
This notebook turns transaction-level bar inventory records into daily demand series, compares a seasonal baseline with a Random Forest forecast, calculates dynamic par levels, and backtests an order-up-to policy.

The analysis is chronological: the first 80% of dates is training data and the final 20% is validation data.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
SOURCE = ROOT / 'Consumption Dataset.xlsx'
PROCESSED = ROOT / 'data' / 'processed' / 'daily_bar_consumption.csv'
PROCESSED.parent.mkdir(parents=True, exist_ok=True)
print('Project root:', ROOT)
print('Source exists:', SOURCE.exists())

## 1. Load, validate, and aggregate the source data

In [ ]:
raw = pd.read_excel(SOURCE, sheet_name='Dataset')
required = ['Date Time Served', 'Bar Name', 'Brand Name', 'Opening Balance (ml)', 'Purchase (ml)', 'Consumed (ml)', 'Closing Balance (ml)']
missing = sorted(set(required) - set(raw.columns))
if missing:
    raise ValueError(f'Missing required columns: {missing}')
raw['Date Time Served'] = pd.to_datetime(raw['Date Time Served'], errors='coerce')
if raw['Date Time Served'].isna().any():
    raise ValueError('Unparseable timestamps found')
numeric = ['Opening Balance (ml)', 'Purchase (ml)', 'Consumed (ml)', 'Closing Balance (ml)']
raw[numeric] = raw[numeric].apply(pd.to_numeric, errors='coerce')
raw['conservation_error_ml'] = raw['Opening Balance (ml)'] + raw['Purchase (ml)'] - raw['Consumed (ml)'] - raw['Closing Balance (ml)']
print('Rows:', len(raw), '| Bars:', raw['Bar Name'].nunique(), '| Brands:', raw['Brand Name'].nunique())
print('Date range:', raw['Date Time Served'].min().date(), 'to', raw['Date Time Served'].max().date())
print('Maximum absolute conservation error (ml):', raw['conservation_error_ml'].abs().max())
raw.head()

In [ ]:
raw['Date'] = raw['Date Time Served'].dt.floor('D')
daily = (raw.groupby(['Date', 'Bar Name', 'Brand Name'], as_index=False)['Consumed (ml)']
         .sum())
pairs = daily[['Bar Name', 'Brand Name']].drop_duplicates()
dates = pd.date_range(daily['Date'].min(), daily['Date'].max(), freq='D')
grid = (pairs.assign(key=1).merge(pd.DataFrame({'Date': dates, 'key': 1}), on='key')
        .drop(columns='key'))
daily_ts = (grid.merge(daily, on=['Date', 'Bar Name', 'Brand Name'], how='left')
            .fillna({'Consumed (ml)': 0.0})
            .sort_values(['Bar Name', 'Brand Name', 'Date'])
            .reset_index(drop=True))
daily_ts.to_csv(PROCESSED, index=False)
print('Daily rows:', len(daily_ts), '| Observed bar-brand pairs:', len(pairs))
print('Zero-demand daily rows:', f"{(daily_ts['Consumed (ml)'].eq(0).mean()):.1%}")
daily_ts.head()

## 2. Exploratory analysis

In [ ]:
daily_ts['Day'] = daily_ts['Date'].dt.day_name()
weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
weekday = daily_ts.groupby('Day')['Consumed (ml)'].mean().reindex(weekday_order)
top_series = (daily_ts.groupby(['Bar Name', 'Brand Name'])['Consumed (ml)'].sum()
              .sort_values(ascending=False).head(10))
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
weekday.plot.bar(ax=axes[0], color='#167d8d', title='Average daily consumption by weekday')
top_series.sort_values().plot.barh(ax=axes[1], color='#e07a5f', title='Top bar-brand series by annual consumption')
axes[0].set_ylabel('Milliliters'); axes[1].set_xlabel('Milliliters')
plt.tight_layout()
plt.show()

In [ ]:
annual = daily_ts.groupby(['Bar Name', 'Brand Name'])['Consumed (ml)'].sum().sort_values(ascending=False)
cum_share = annual.cumsum() / annual.sum()
abc = pd.cut(cum_share, bins=[0, 0.80, 0.95, 1.0], labels=['A', 'B', 'C'], include_lowest=True)
abc_table = annual.rename('annual_consumption_ml').to_frame()
abc_table['cumulative_share'] = cum_share
abc_table['ABC_class'] = abc
abc_table.head(10)

## 3. Forecasting: seasonal baseline versus Random Forest

In [ ]:
model_df = daily_ts.copy()
grouped = model_df.groupby(['Bar Name', 'Brand Name'], sort=False)['Consumed (ml)']
for lag in [1, 7, 14]:
    model_df[f'lag_{lag}'] = grouped.shift(lag)
model_df['rolling_mean_7'] = grouped.shift(1).transform(lambda x: x.rolling(7, min_periods=3).mean())
model_df['rolling_std_7'] = grouped.shift(1).transform(lambda x: x.rolling(7, min_periods=3).std())
model_df['dayofweek'] = model_df['Date'].dt.dayofweek
model_df['is_weekend'] = model_df['dayofweek'].isin([4, 5, 6]).astype(int)
cutoff = model_df['Date'].min() + (model_df['Date'].max() - model_df['Date'].min()) * 0.8
feature_cols = ['lag_1', 'lag_7', 'lag_14', 'rolling_mean_7', 'rolling_std_7', 'dayofweek', 'is_weekend', 'Bar Name', 'Brand Name']
model_df = model_df.dropna(subset=['lag_1', 'lag_7', 'lag_14', 'rolling_mean_7', 'rolling_std_7']).copy()
train = model_df[model_df['Date'] < cutoff].copy()
test = model_df[model_df['Date'] >= cutoff].copy()
X_train, y_train = train[feature_cols], train['Consumed (ml)']
X_test, y_test = test[feature_cols], test['Consumed (ml)']
categorical = ['Bar Name', 'Brand Name']
preprocess = ColumnTransformer([('categorical', OneHotEncoder(handle_unknown='ignore'), categorical)], remainder='passthrough')
rf = Pipeline([('preprocess', preprocess), ('model', RandomForestRegressor(n_estimators=60, min_samples_leaf=3, random_state=42, n_jobs=-1))])
rf.fit(X_train, y_train)
test['seasonal_naive_7'] = test['lag_7'].clip(lower=0)
test['random_forest'] = np.maximum(0, rf.predict(X_test))
print('Train rows:', len(train), '| Validation rows:', len(test), '| Cutoff:', cutoff.date())

In [ ]:
def wape(actual, forecast):
    denominator = np.abs(actual).sum()
    return np.abs(actual - forecast).sum() / denominator if denominator else np.nan

metrics = pd.DataFrame({
    'MAE_ml': [mean_absolute_error(y_test, test['seasonal_naive_7']), mean_absolute_error(y_test, test['random_forest'])],
    'RMSE_ml': [mean_squared_error(y_test, test['seasonal_naive_7']) ** 0.5, mean_squared_error(y_test, test['random_forest']) ** 0.5],
    'WAPE': [wape(y_test.to_numpy(), test['seasonal_naive_7'].to_numpy()), wape(y_test.to_numpy(), test['random_forest'].to_numpy())]
}, index=['Seasonal naive (7 days)', 'Random Forest'])
metrics

## 4. Dynamic par levels and inventory simulation

In [ ]:
def compute_par_level(predicted_daily_demand, std_daily_demand, lead_time_days=2, service_level_z=1.645):
    lead_time_demand = max(0.0, predicted_daily_demand) * lead_time_days
    safety_stock = service_level_z * max(0.0, std_daily_demand) * np.sqrt(lead_time_days)
    return lead_time_demand + safety_stock, safety_stock

def simulate_inventory(actual_demand, par_level, lead_time=2, initial_stock=None):
    initial_stock = par_level if initial_stock is None else initial_stock
    stock = float(initial_stock)
    pending = []
    stockout_days = 0
    lost_volume = 0.0
    history = []
    for demand in actual_demand:
        arriving = sum(quantity for days, quantity in pending if days <= 1)
        pending = [(days - 1, quantity) for days, quantity in pending if days > 1]
        stock += arriving
        fulfilled = min(stock, float(demand))
        lost = float(demand) - fulfilled
        stock -= fulfilled
        if lost > 0:
            stockout_days += 1
            lost_volume += lost
        inventory_position = stock + sum(quantity for _, quantity in pending)
        if inventory_position < par_level:
            pending.append((lead_time, par_level - inventory_position))
        history.append(stock)
    return {'stockout_days': stockout_days, 'lost_volume_ml': lost_volume, 'average_inventory_ml': np.mean(history), 'history': history}

# Use the model's validation predictions to estimate one policy par per bar-brand series.
series_stats = (train.groupby(['Bar Name', 'Brand Name'])
                .agg(predicted_daily_demand=('lag_7', 'mean'), std_daily_demand=('Consumed (ml)', 'std'))
                .reset_index())
series_stats['std_daily_demand'] = series_stats['std_daily_demand'].fillna(0)
series_stats[['par_level_ml', 'safety_stock_ml']] = series_stats.apply(lambda r: pd.Series(compute_par_level(r['predicted_daily_demand'], r['std_daily_demand'])), axis=1)
series_stats.head()

In [ ]:
simulation_rows = []
for keys, series in test.groupby(['Bar Name', 'Brand Name']):
    stat = series_stats[(series_stats['Bar Name'] == keys[0]) & (series_stats['Brand Name'] == keys[1])].iloc[0]
    result = simulate_inventory(series['Consumed (ml)'].to_numpy(), stat['par_level_ml'])
    simulation_rows.append({'Bar Name': keys[0], 'Brand Name': keys[1], 'par_level_ml': stat['par_level_ml'], **{k: v for k, v in result.items() if k != 'history'}})
simulation = pd.DataFrame(simulation_rows)
print('Aggregate validation policy results:')
display(simulation[['stockout_days', 'lost_volume_ml', 'average_inventory_ml']].sum(numeric_only=True).to_frame('total'))
simulation.sort_values('lost_volume_ml', ascending=False).head(10)

In [ ]:
selected = test.groupby(['Bar Name', 'Brand Name'])['Consumed (ml)'].sum().idxmax()
selected_test = test[(test['Bar Name'] == selected[0]) & (test['Brand Name'] == selected[1])].copy()
selected_stat = series_stats[(series_stats['Bar Name'] == selected[0]) & (series_stats['Brand Name'] == selected[1])].iloc[0]
selected_sim = simulate_inventory(selected_test['Consumed (ml)'].to_numpy(), selected_stat['par_level_ml'])
plt.figure(figsize=(14, 5))
plt.plot(selected_test['Date'], selected_sim['history'], label='Closing stock', color='#167d8d')
plt.axhline(selected_stat['par_level_ml'], color='#e07a5f', linestyle='--', label='Par level')
plt.title(f'Validation inventory path: {selected[0]} / {selected[1]}')
plt.ylabel('Milliliters'); plt.xlabel('Date'); plt.legend(); plt.tight_layout(); plt.show()
print('Selected policy:', {k: selected_sim[k] for k in ['stockout_days', 'lost_volume_ml', 'average_inventory_ml']})

## 5. Interpretation and deployment notes

Use the metric table to choose the forecast model, then use the simulation to choose a service level. A lower forecast error does not automatically mean lower stockouts if lead time or safety stock assumptions are wrong. In production, monitor forecast WAPE, stockout days, lost volume, holding stock, supplier lead-time variance, and conservation/data-quality errors.